In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DelayLayerLogic(nn.Module):
    def __init__(self, in_features, out_features, max_delay):
        super(DelayLayerLogic, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.max_delay = max_delay

        # Weights & Delays
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.delay = nn.Parameter(torch.rand(out_features, in_features) * max_delay)

    def forward(self, x):
        """
        x shape: (T, B, C)  where C = in_features
        Output shape: (T, B, out_features)
        """
        T, B, C = x.shape
        device = x.device

        # Clamp delays
        delay = torch.clamp(self.delay, 0.0, self.max_delay)  # (Out, In)

        floor_d = torch.floor(delay).long()                    # (Out, In)
        ceil_d = torch.ceil(delay).long()
        alpha = delay - floor_d.float()                        # (Out, In)

        # Time indices
        t = torch.arange(T, device=device).view(T, 1, 1, 1)   # (T, 1, 1, 1)

        # Compute delayed indices
        t_floor = torch.clamp(t - floor_d.view(1, 1, *floor_d.shape), 0, T - 1)
        t_ceil = torch.clamp(t - ceil_d.view(1, 1, *ceil_d.shape), 0, T - 1)

        # Gather with advanced indexing
        # Expand x for broadcasting: (T, B, 1, C)
        x_expanded = x.unsqueeze(2)  # (T, B, 1, C)

        # Get floor and ceil values
        # Note: We use gather for correct indexing
        idx_floor = t_floor.expand(-1, B, -1, -1)  # (T, B, Out, In)
        idx_ceil = t_ceil.expand(-1, B, -1, -1)

        # Gather (this is the key vectorized part)
        x_floor = torch.gather(x_expanded.expand(-1, -1, self.out_features, -1),
                              dim=0, index=idx_floor)
        x_ceil = torch.gather(x_expanded.expand(-1, -1, self.out_features, -1),
                             dim=0, index=idx_ceil)

        # Linear interpolation
        alpha_exp = alpha.view(1, 1, *alpha.shape)  # (1, 1, Out, In)
        interpolated = (1.0 - alpha_exp) * x_floor + alpha_exp * x_ceil

        # Apply weights and sum
        weighted = interpolated * self.weight.view(1, 1, *self.weight.shape)
        output = torch.sum(weighted, dim=-1)  # (T, B, Out)

        return output

In [ ]:
import torch
import torch.nn as nn
from spikingjelly.activation_based import neuron, surrogate, functional



class DeepDelaySNN(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, max_delay):
        super(DeepDelaySNN, self).__init__()

        surrogate_function = surrogate.ATan()

        self.layer1 = DelayLayerLogic(in_features, hidden_features, max_delay)
        self.lif1 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate_function)
        self.dropout = nn.Dropout(p=0.05)


        self.layer2 = DelayLayerLogic(hidden_features, out_features, max_delay)
        self.lif2 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate_function)

    def forward(self, x):

        out1 = self.layer1(x)
        spike1 = self.lif1(out1)
        spike1_dropped = self.dropout(spike1)

        out2 = self.layer2(spike1_dropped)
        spike2 = self.lif2(out2)

        return spike2

In [ ]:
# import os
# import gzip
# import shutil
# import urllib.request

# # مسیر ذخیره
# data_dir = "./datasets_ssc"
# os.makedirs(data_dir, exist_ok=True)

# base_url = "https://zenkelab.org/datasets/"

# files = [
#     "ssc_train.h5.gz",
#     "ssc_valid.h5.gz",
#     "ssc_test.h5.gz"
# ]

# for fname in files:
#     gz_path = os.path.join(data_dir, fname)
#     h5_path = gz_path[:-3]   # حذف .gz

#     if os.path.exists(h5_path):
#         print(f"{h5_path} از قبل وجود دارد، رد شد.")
#         continue

#     print(f"در حال دانلود {fname} ...")
#     url = base_url + fname
#     urllib.request.urlretrieve(url, gz_path)
#     print(f"دانلود {fname} تمام شد.")

#     # استخراج فایل
#     print(f"در حال استخراج {fname} ...")
#     with gzip.open(gz_path, 'rb') as f_in:
#         with open(h5_path, 'wb') as f_out:
#             shutil.copyfileobj(f_in, f_out)

#     # حذف فایل فشرده برای صرفه‌جویی در فضا
#     os.remove(gz_path)
#     print(f"{h5_path} آماده شد.\n")

# print("همه فایل‌های SSC با موفقیت دانلود و استخراج شدند.")

In [ ]:
import os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class SSC(Dataset):
    def __init__(self, root, split='train', time_steps=20):
        self.root = root
        self.split = split
        self.time_steps = time_steps

        file_map = {
            'train': 'ssc_train.h5',
            'val':   'ssc_valid.h5',
            'test':  'ssc_test.h5'
        }

        if split not in file_map:
            raise ValueError("split باید یکی از 'train', 'val', 'test' باشد")

        self.h5_path = os.path.join(root, file_map[split])

        if not os.path.exists(self.h5_path):
            raise FileNotFoundError(f"فایل {self.h5_path} پیدا نشد.")

        with h5py.File(self.h5_path, 'r') as f:
            self.units = f['spikes']['units'][:]
            self.times = f['spikes']['times'][:]
            self.labels = f['labels'][:]

        print(f"SSC {split.upper()} samples: {len(self.labels)}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        units = self.units[index].astype(np.int64)
        times = self.times[index].astype(np.float64)
        label = int(self.labels[index])

        frames = np.zeros((self.time_steps, 700), dtype=np.float32)

        if len(times) > 0:
            t_min, t_max = times.min(), times.max()
            if t_max > t_min:
                times = (times - t_min) / (t_max - t_min)
            else:
                times = np.zeros_like(times)

            bin_idx = np.clip((times * self.time_steps).astype(np.int64), 0, self.time_steps - 1)

            for t, c in zip(bin_idx, units):
                if 0 <= c < 700:
                    frames[t, c] += 1.0

        frames = np.clip(frames, 0, 1)
        return torch.from_numpy(frames), torch.tensor(label, dtype=torch.long)


def get_ssc_dataloader(data_dir, batch_size, time_steps=20, split='train'):
    dataset = SSC(root=data_dir, split=split, time_steps=time_steps)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == 'train'),
        drop_last=(split == 'train'),
        num_workers=0,
        pin_memory=True
    )
    return loader

In [ ]:
import torch
import torch.nn as nn
from spikingjelly.activation_based import functional, monitor, neuron
import os

def train_weight_only_fixed_delay_snn(model, train_loader, val_loader, test_loader, epochs, device, seed):

    os.makedirs(f"./outputs_fixed_delay_seed{seed}", exist_ok=True)

    print("\n1- Starting Weight-Only Learning...")

    for param in model.parameters():
        param.requires_grad = True

    weight_parameters = [p for n, p in model.named_parameters() if 'weight' in n]
    delay_parameters = [p for n, p in model.named_parameters() if 'delay' in n]

    for p in delay_parameters:
        p.requires_grad = False


    optimizer = torch.optim.Adam([{'params': weight_parameters, 'lr': 0.01, 'weight_decay': 4e-4}])

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.2,
        patience=7,
        cooldown=1,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss()

    spike_monitor = monitor.OutputMonitor(model, neuron.LIFNode)
    model.to(device)

    history = {
        'train_loss': [], 'train_acc': [], 'train_spikes': [],
        'val_loss': [], 'val_acc': [], 'val_spikes': []
    }

    best_val_acc = 0.0
    best_val_loss = 100.0
    print("\nStarting Training Loop (Weights Tuning)...")

    try:
        for epoch in range(epochs):
            # Train Phase
            model.train()
            total_train_loss, train_correct, train_total, total_train_spikes = 0, 0, 0, 0

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                functional.reset_net(model)
                optimizer.zero_grad()


                if inputs.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                    B, T, C, H, W = inputs.shape
                    inputs = inputs.view(B, T, C * H * W).permute(1, 0, 2)
                elif inputs.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                    B, T, C, H = inputs.shape
                    inputs = inputs.view(B, T, C * H).permute(1, 0, 2)
                elif inputs.dim() == 3:        # (B, T, C)
                    inputs = inputs.permute(1, 0, 2)
                else:
                    raise ValueError(f"Unexpected input shape: {frames.shape}")

                inputs, targets = inputs.to(device), targets.to(device)

                spike_monitor.enable()
                out_spikes = model(inputs)

                batch_spikes = sum(r.sum().item() for r in spike_monitor.records)
                total_train_spikes += (batch_spikes / targets.size(0))
                spike_monitor.records.clear()
                spike_monitor.disable()

                mean_firing_rate = out_spikes.mean(dim=0)
                loss = criterion(mean_firing_rate, targets)
                loss.backward()
                optimizer.step()

                _, predicted = torch.max(mean_firing_rate, dim=1)
                train_total += targets.size(0)
                train_correct += (predicted == targets).sum().item()


                total_train_loss += loss.item()

            epoch_train_loss = total_train_loss / len(train_loader)
            epoch_train_acc = (train_correct / train_total) * 100
            epoch_train_avg_spikes = total_train_spikes / len(train_loader)

            history['train_loss'].append(epoch_train_loss)
            history['train_acc'].append(epoch_train_acc)
            history['train_spikes'].append(epoch_train_avg_spikes)

            # Validation Phase
            model.eval()
            total_val_loss, val_correct, val_total, total_val_spikes = 0, 0, 0, 0

            with torch.no_grad():
                for frames, labels in val_loader:
                    functional.reset_net(model)
                    # تبدیل به (T, B, C=64)
                    if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                        B, T, C, H, W = frames.shape
                        frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
                    elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                        B, T, C, H = frames.shape
                        frames = frames.view(B, T, C * H).permute(1, 0, 2)
                    elif frames.dim() == 3:        # (B, T, C)
                        frames = frames.permute(1, 0, 2)
                    else:
                        raise ValueError(f"Unexpected input shape: {frames.shape}")

                    frames, labels = frames.to(device), labels.to(device)

                    spike_monitor.enable()
                    outputs = model(frames)

                    batch_val_spikes = sum(r.sum().item() for r in spike_monitor.records)
                    total_val_spikes += (batch_val_spikes / labels.size(0))
                    spike_monitor.records.clear()
                    spike_monitor.disable()

                    mean_out = outputs.mean(dim=0)
                    v_loss = criterion(mean_out, labels)
                    total_val_loss += v_loss.item()

                    pred = mean_out.argmax(dim=1)
                    val_correct += (pred == labels).sum().item()
                    val_total += labels.numel()



            epoch_val_loss = total_val_loss / len(val_loader)
            epoch_val_acc = (val_correct / val_total) * 100
            epoch_val_avg_spikes = total_val_spikes / len(val_loader)

            scheduler.step(epoch_val_acc)

            history['val_loss'].append(epoch_val_loss)
            history['val_acc'].append(epoch_val_acc)
            history['val_spikes'].append(epoch_val_avg_spikes)

            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Train Loss={epoch_train_loss:.4f}, Acc={epoch_train_acc:.2f}%, Spikes={epoch_train_avg_spikes:.1f} || "
                f"Val Loss={epoch_val_loss:.4f}, Acc={epoch_val_acc:.2f}%, Spikes={epoch_val_avg_spikes:.1f}"
            )

            if epoch_val_acc > best_val_acc:
                best_val_acc = epoch_val_acc
                torch.save(model.state_dict(), f"./outputs_fixed_delay_seed{seed}/best_val_acc_model.pth")
                print(f"--> [SAVE] Checkpoint Archived! Best Val Acc: {best_val_acc:.2f}%")

    except KeyboardInterrupt:
        print("\nTraining interrupted by user.")

    log_path = f"./outputs_fixed_delay_seed{seed}/fixed_delay_logs.txt"
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("Epoch,Train_Loss,Train_Acc,Train_Spikes,Val_Loss,Val_Acc,Val_Spikes\n")
        for i in range(len(history['train_loss'])):
            f.write(
                f"{i+1},{history['train_loss'][i]:.4f},{history['train_acc'][i]:.2f},{history['train_spikes'][i]:.1f},"
                f"{history['val_loss'][i]:.4f},{history['val_acc'][i]:.2f},{history['val_spikes'][i]:.1f}\n"
            )
    print(f"All learning logs archived at '{log_path}'.")

    # Test Phase
    print("\n=== Training Finished. Loading Best Acc Model for Final Test Evaluation... ===")
    if os.path.exists(f"./outputs_fixed_delay_seed{seed}/best_val_acc_model.pth"):
        model.load_state_dict(torch.load(f"./outputs_fixed_delay_seed{seed}/best_val_acc_model.pth"))

    model.eval()
    total_test_loss, test_correct, test_total, total_test_spikes = 0, 0, 0, 0

    with torch.no_grad():
        for frames, labels in test_loader:
            functional.reset_net(model)
            # تبدیل به (T, B, C=64)
            if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                B, T, C, H, W = frames.shape
                frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
            elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                B, T, C, H = frames.shape
                frames = frames.view(B, T, C * H).permute(1, 0, 2)
            elif frames.dim() == 3:        # (B, T, C)
                frames = frames.permute(1, 0, 2)
            else:
                raise ValueError(f"Unexpected input shape: {frames.shape}")

            frames, labels = frames.to(device), labels.to(device)

            spike_monitor.enable()
            outputs = model(frames)

            batch_test_spikes = sum(r.sum().item() for r in spike_monitor.records)
            total_test_spikes += (batch_test_spikes / labels.size(0))
            spike_monitor.records.clear()
            spike_monitor.disable()

            mean_out = outputs.mean(dim=0)
            t_loss = criterion(mean_out, labels)
            pred = mean_out.argmax(dim=1)
            total_test_loss += t_loss.item()
            test_correct += (pred == labels).sum().item()
            test_total += labels.numel()


    final_test_loss = total_test_loss / len(test_loader)
    final_test_acc = (test_correct / test_total) * 100
    final_test_spikes = total_test_spikes / len(test_loader)

    print(f"\n🚀 [FINAL RESULTS - FIXED DELAY] 🚀")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Final Test Loss (Unseen Data): {final_test_loss:.4f}")
    print(f"Final Test Accuracy (Unseen Data): {final_test_acc:.2f}%")
    print(f"Final Test Average Spike Count: {final_test_spikes:.1f}")

    with open(f"./outputs_fixed_delay_seed{seed}/final_test_acc_report.txt", "w", encoding="utf-8") as f:
        f.write(f"Best Val Accuracy: {best_val_acc:.2f}%\n")
        f.write(f"Final Test Loss: {final_test_loss:.4f}\n")
        f.write(f"Final Test Accuracy: {final_test_acc:.2f}%\n")
        f.write(f"Final Test Average Spikes: {final_test_spikes:.1f}\n")

In [ ]:
import os
import torch
import torch.nn as nn
from spikingjelly.activation_based import functional, monitor

def quantize_tensor(tensor, bits=8):
    if bits == 32:
        return tensor.clone()
    qmin = -(2 ** (bits - 1)) + 1
    qmax = (2 ** (bits - 1)) - 1
    max_val = torch.max(torch.abs(tensor))
    if max_val == 0:
        return tensor.clone()
    scale = max_val / qmax
    quantized = torch.round(tensor / scale)
    quantized = torch.clamp(quantized, qmin, qmax)
    return quantized * scale

def run_test_for_checkpoint(model_fn, test_loader, checkpoint_path, bits, device):
    model = model_fn()
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    if bits < 32:
        with torch.no_grad():
            for name, param in model.named_parameters():
                if 'weight' in name:
                    param.copy_(quantize_tensor(param.data, bits=bits))

    spike_monitor = monitor.OutputMonitor(model, neuron.LIFNode)
    test_correct, test_total, total_test_loss, total_test_spikes = 0, 0, 0, 0

    with torch.no_grad():
        for frames, labels in test_loader:
            functional.reset_net(model)
            # تبدیل به (T, B, C=64)
            if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                B, T, C, H, W = frames.shape
                frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
            elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                B, T, C, H = frames.shape
                frames = frames.view(B, T, C * H).permute(1, 0, 2)
            elif frames.dim() == 3:        # (B, T, C)
                frames = frames.permute(1, 0, 2)
            else:
                raise ValueError(f"Unexpected input shape: {frames.shape}")

            frames, labels = frames.to(device), labels.to(device)

            spike_monitor.enable()
            outputs = model(frames)

            batch_test_spikes = sum(r.sum().item() for r in spike_monitor.records)
            total_test_spikes += (batch_test_spikes / labels.size(0))

            spike_monitor.disable()
            spike_monitor.records.clear()

            mean_out = outputs.mean(dim=0)
            t_loss = criterion(mean_out, labels)
            total_test_loss += t_loss.item()
            pred = mean_out.argmax(dim=1)
            test_correct += (pred == labels).sum().item()
            test_total += labels.numel()


    final_test_loss = total_test_loss / len(test_loader)
    final_test_acc = (test_correct / test_total) * 100
    final_test_spikes = total_test_spikes / len(test_loader)
    return final_test_loss, final_test_acc, final_test_spikes

def compare_joint_vs_fixed_quantization(model_fn, ckpt, test_loader, device='cuda', seed=42):
    fixed_ckpt = ckpt

    bits_options = [32, 8, 4, 2]
    results = {"Fixed-Delay": {}}

    print("=== Starting Comprehensive Quantization Comparison ===")

    for bits in bits_options:
        print(f"\n[Evaluating {bits}-bit Precision...]")


        if os.path.exists(fixed_ckpt):
            loss, acc, spks = run_test_for_checkpoint(model_fn, test_loader, fixed_ckpt, bits, device)
            results["Fixed-Delay"][bits] = (loss, acc, spks)
            print(f"-> Fixed-Delay Model Acc: {acc:.2f}%"
            f" Loss: {loss:.2f}"
            f" avg spikes: {spks:.2f}")
        else:
            results["Fixed-Delay"][bits] = None

    print("\n" + "="*80)
    print("FINAL QUANTIZATION ABLATION TABLE")
    print("="*80)
    print(f"{'Bits':<10}{'Loss':<12}{'Acc (%)':<12}{'Avg Spikes':<12}")
    print("-"*80)

    for bits in bits_options:
        result = results["Fixed-Delay"][bits]

        if result is not None:
            loss, acc, spks = result
            print(f"{bits:<10}{loss:<12.4f}{acc:<12.2f}{spks:<12.2f}")
        else:
            print(f"{bits:<10}{'N/A':<12}{'N/A':<12}{'N/A':<12}")

    print("="*80)

    with open(f"./outputs_fixed_delay_seed{seed}/quantization_comparison_report.txt",
              "w", encoding="utf-8") as f:
        f.write("Bits,Loss,Accuracy,AvgSpikes\n")

        for bits in bits_options:
            result = results["Fixed-Delay"][bits]

            if result is not None:
                loss, acc, spks = result
                f.write(f"{bits},{loss:.6f},{acc:.2f},{spks:.2f}\n")
            else:
                f.write(f"{bits},N/A,N/A,N/A\n")

In [ ]:
def train_delay_recovery_from_quantized_weights(
        model,
        ckpt,
        train_loader,
        val_loader,
        test_loader,
        epochs,
        bits,
        max_delay,
        device,
        seed):

    model.load_state_dict(
        torch.load(ckpt, map_location=device)
    )

    print(f"\n=== Delay Recovery Experiment ({bits}-bit) ===")

    # Quantize weights once
    with torch.no_grad():
        for name, param in model.named_parameters():
            if "weight" in name:
                param.copy_(quantize_tensor(param.data, bits))

    # Freeze weights
    for name, param in model.named_parameters():

        if "weight" in name:
            param.requires_grad = False

        elif "delay" in name:
            param.requires_grad = True

    delay_params = [
        p for n,p in model.named_parameters()
        if "delay" in n
    ]

    optimizer = torch.optim.Adam(
        delay_params,
        lr=0.1
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=5)

    criterion = nn.CrossEntropyLoss()

    model.to(device)

    best_val_acc = -1.0

    spike_monitor = monitor.OutputMonitor(model, neuron.LIFNode)

    history = {
        'train_loss': [], 'train_acc': [], 'train_spikes': [],
        'val_loss': [], 'val_acc': [], 'val_spikes': []
    }

    for epoch in range(epochs):

        model.train()

        correct = 0
        total = 0
        total_train_loss = 0
        total_train_spikes = 0

        for frames, labels in train_loader:
            functional.reset_net(model)
            optimizer.zero_grad()

            # تبدیل به (T, B, C=64)
            if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                B, T, C, H, W = frames.shape
                frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
            elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                B, T, C, H = frames.shape
                frames = frames.view(B, T, C * H).permute(1, 0, 2)
            elif frames.dim() == 3:        # (B, T, C)
                frames = frames.permute(1, 0, 2)
            else:
                raise ValueError(f"Unexpected input shape: {frames.shape}")

            frames = frames.to(device)
            labels = labels.to(device)

            spike_monitor.enable()
            outputs = model(frames)

            batch_spikes = sum(r.sum().item() for r in spike_monitor.records)
            total_train_spikes += (batch_spikes / labels.size(0))
            spike_monitor.records.clear()
            spike_monitor.disable()

            mean_out = outputs.mean(dim=0)

            loss = criterion(mean_out, labels)

            loss.backward()

            optimizer.step()

            with torch.no_grad():
                for n,p in model.named_parameters():
                    if "delay" in n:
                        p.clamp_(0,max_delay)

            pred = mean_out.argmax(dim=1)

            correct += (pred == labels).sum().item()
            total += labels.numel()


            total_train_loss += loss.item()

        epoch_train_acc = 100 * correct / total
        epoch_train_loss = total_train_loss / len(train_loader)
        epoch_train_avg_spikes = total_train_spikes / len(train_loader)

        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['train_spikes'].append(epoch_train_avg_spikes)

        # Validation

        model.eval()

        total_val_loss, val_correct, val_total, total_val_spikes = 0, 0, 0, 0

        with torch.no_grad():

            for frames, labels in val_loader:
                functional.reset_net(model)
                # تبدیل به (T, B, C=64)
                if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                    B, T, C, H, W = frames.shape
                    frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
                elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                    B, T, C, H = frames.shape
                    frames = frames.view(B, T, C * H).permute(1, 0, 2)
                elif frames.dim() == 3:        # (B, T, C)
                    frames = frames.permute(1, 0, 2)
                else:
                    raise ValueError(f"Unexpected input shape: {frames.shape}")

                frames = frames.to(device)
                labels = labels.to(device)

                spike_monitor.enable()

                outputs = model(frames)

                batch_val_spikes = sum(r.sum().item() for r in spike_monitor.records)
                total_val_spikes += (batch_val_spikes / labels.size(0))
                spike_monitor.records.clear()
                spike_monitor.disable()

                outputs = outputs.mean(dim=0)
                v_loss = criterion(outputs, labels)
                total_val_loss += v_loss.item()
                pred = outputs.argmax(dim=1)

                val_correct += (pred == labels).sum().item()
                val_total += labels.numel()

        epoch_val_loss = total_val_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_total) * 100
        epoch_val_avg_spikes = total_val_spikes / len(val_loader)

        scheduler.step(epoch_val_acc)

        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['val_spikes'].append(epoch_val_avg_spikes)

        if epoch_val_acc > best_val_acc:
          best_val_acc = epoch_val_acc
          os.makedirs(f"./outputs_quant_delay_seed{seed}", exist_ok=True)
          torch.save(
              model.state_dict(),
              f"./outputs_quant_delay_seed{seed}/best_delay_recovery_{bits}bits.pth"
          )


        print(
            f"Epoch {epoch+1}/{epochs} "
            f"| Train Acc = {epoch_train_acc:.2f}% "
            f"| Train Loss = {epoch_train_loss:.4f}"
            f"| Val Acc = {epoch_val_acc:.2f}% "
            f"| Val Loss = {epoch_val_loss:.4f}"
            f"| Best Val = {best_val_acc:.2f}%"
        )

    log_path = f"./outputs_quant_delay_seed{seed}/quant_delay_logs_{bits}bit.txt"
    os.makedirs(f"./outputs_quant_delay_seed{seed}", exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("Epoch,Train_Loss,Train_Acc,Train_Spikes,Val_Loss,Val_Acc,Val_Spikes\n")
        for i in range(len(history['train_loss'])):
            f.write(
                f"{i+1},{history['train_loss'][i]:.4f},{history['train_acc'][i]:.2f},{history['train_spikes'][i]:.1f},"
                f"{history['val_loss'][i]:.4f},{history['val_acc'][i]:.2f},{history['val_spikes'][i]:.1f}\n"
            )
    print(f"All learning logs archived at '{log_path}'.")

    # Final Test

    model.load_state_dict(
      torch.load(
          f"./outputs_quant_delay_seed{seed}/best_delay_recovery_{bits}bits.pth",
          map_location=device
      )
    )

    model.eval()

    total_test_loss = 0
    test_correct = 0
    test_total = 0
    total_test_spikes = 0

    with torch.no_grad():

        for frames, labels in test_loader:
            functional.reset_net(model)
            # تبدیل به (T, B, C=64)
            if frames.dim() == 5:          # (B, T, C, H, W)  مثلاً (B, T, 1, 64, 1)
                B, T, C, H, W = frames.shape
                frames = frames.view(B, T, C * H * W).permute(1, 0, 2)
            elif frames.dim() == 4:        # (B, T, C, H)     مثلاً (B, T, 1, 64)
                B, T, C, H = frames.shape
                frames = frames.view(B, T, C * H).permute(1, 0, 2)
            elif frames.dim() == 3:        # (B, T, C)
                frames = frames.permute(1, 0, 2)
            else:
                raise ValueError(f"Unexpected input shape: {frames.shape}")

            frames = frames.to(device)
            labels = labels.to(device)

            spike_monitor.enable()

            outputs = model(frames)

            batch_test_spikes = sum(r.sum().item() for r in spike_monitor.records)
            total_test_spikes += (batch_test_spikes / labels.size(0))
            spike_monitor.records.clear()
            spike_monitor.disable()

            mean_out = outputs.mean(dim=0)
            test_loss_batch = criterion(mean_out, labels)
            pred = mean_out.argmax(dim=1)

            total_test_loss += test_loss_batch.item()
            test_correct += (pred == labels).sum().item()
            test_total += labels.numel()

    test_loss = total_test_loss / len(test_loader)
    test_acc = 100 * test_correct / test_total
    final_test_spikes = total_test_spikes / len(test_loader)
    spike_monitor.remove_hooks()
    print(
        f"\nRecovered Accuracy ({bits}-bit + Delay Learning): "
        f"{test_acc:.2f}% | "
        f"Loss: {test_loss:.4f} | "
        f"Avg Spike Count: {final_test_spikes:.1f}"
    )
    with open(
        f"./outputs_quant_delay_seed{seed}/result_{bits}bit.txt",
        "w"
    ) as f:
        f.write(f"Best Val Accuracy: {best_val_acc:.4f}\n")
        f.write(f"Final Test Accuracy: {test_acc:.4f}\n")
        f.write(f"Final Test Loss: {test_loss:.4f}\n")
        f.write(f"Final Test Avg Spikes: {final_test_spikes:.4f}\n")
    return test_loss, test_acc, final_test_spikes

In [ ]:
import torch
import os
import random
import numpy as np
from torch.utils.data import SubsetRandomSampler


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def main():
    # Hardware settings
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Hyperparameters
    max_delay = 5.0
    epochs = 100
    batch_size = 64
    time_steps = 20

    data_dir = "./datasets_ssc"
    os.makedirs(data_dir, exist_ok=True)

    print("Loading SSC Dataset...")

    train_loader = get_ssc_dataloader(data_dir, batch_size, time_steps, split='train')
    val_loader   = get_ssc_dataloader(data_dir, batch_size, time_steps, split='val')
    test_loader  = get_ssc_dataloader(data_dir, batch_size, time_steps, split='test')


    in_features = 700
    out_features = 35
    hidden_features = 128
    seed_num = [42, 43, 44]

    for i in seed_num:
        set_seed(i)
        model = DeepDelaySNN(in_features, hidden_features, out_features, max_delay)
        os.makedirs(f"./init_values_seed{i}", exist_ok=True)
        torch.save(model.state_dict(), f"./init_values_seed{i}/initial_full_model.pth")
        print("--> [SAVE] Initial matrices archived. Ready for comparative analysis.")

        print(f"\nTrain Weight-only | SEED = {i}")
        train_weight_only_fixed_delay_snn(model, train_loader, val_loader, test_loader, epochs, device, i)



    print("Preparing components for Quantization Analysis...")
    for i in seed_num:
        print(f"\n=== Quantization Analysis for Seed {i} ===")

        model_functional_closer = lambda: DeepDelaySNN(
            in_features=700,
            hidden_features=128,
            out_features=35,
            max_delay=5.0
        )

        ckpt = f"./outputs_fixed_delay_seed{i}/best_val_acc_model.pth"

        compare_joint_vs_fixed_quantization(
            model_fn=model_functional_closer,
            ckpt=ckpt,
            test_loader=test_loader,
            device=device,
            seed=i
        )

    # ------------------ Delay Recovery Phase ------------------
    for i in seed_num:
        print(f"\n==========================================")
        print(f"⌛ Delay Learning with Seed {i}")
        print(f"==========================================")

        ckpt = f"./outputs_fixed_delay_seed{i}/best_val_acc_model.pth"

        model_recovery = DeepDelaySNN(in_features, hidden_features, out_features, max_delay)


        target_bits = [2, 4, 8, 32]

        recovery_results = {}

        for b in target_bits:
            print(f"\n---> Starting Delay Recovery for {b}-bit Quantized Weights...")

            acc_nbit_delay = train_delay_recovery_from_quantized_weights(
                model=model_recovery,
                ckpt=ckpt,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                epochs=30,
                bits=b,
                max_delay=max_delay,
                device=device,
                seed=i
            )

            recovery_results[b] = acc_nbit_delay

        print(f"Results for Seed {i} Recovery: {recovery_results}")




if __name__ == '__main__':
    main()

Using device: cuda
Loading SSC Dataset...
SSC TRAIN samples: 75466
SSC VAL samples: 9981
SSC TEST samples: 20382
--> [SAVE] Initial matrices archived. Ready for comparative analysis.

Train Weight-only | SEED = 42

1- Starting Weight-Only Learning...

Starting Training Loop (Weights Tuning)...
Epoch 01/100 | Train Loss=3.4370, Acc=8.35%, Spikes=626.8 || Val Loss=3.4017, Acc=8.68%, Spikes=521.5
--> [SAVE] Checkpoint Archived! Best Val Acc: 8.68%

Training interrupted by user.
All learning logs archived at './outputs_fixed_delay_seed42/fixed_delay_logs.txt'.

=== Training Finished. Loading Best Acc Model for Final Test Evaluation... ===
